In [ ]:
# Core dependencies
!pip install spacy sentence-transformers numpy sqlalchemy

# Download spaCy model
!python -m spacy download en_core_web_lg

# Optional: For advanced skill extraction
!pip install skillNer pyPDF2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 3.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 19.3 MB/s eta 0:00:00
  Created wheel for skillNer: filename=skillNer-1.0.3-py3-none-any.whl size=25625 sha256=725ce454c4f171c65c64339622907289f8fae8364b0864be95f4b33b2a3ef294
  Stored in directory: /root/.cache/pip/wheels/05/61/2f/331f7cbd5980aceaec089ca82ce49c282523d8365bf99e8013
Successfully built skillNer


In [ ]:
import re
from collections import Counter, defaultdict
import logging
from typing import List, Dict, Any, Set, Tuple, Optional, Union

import numpy as np
import spacy
from spacy.matcher import PhraseMatcher
from sentence_transformers import SentenceTransformer, util

# optional dependencies
try:
    from skillNer.skill_extractor_class import SkillExtractor
    SKILL_NER_AVAILABLE = True
except ImportError:
    SkillExtractor = None
    SKILL_NER_AVAILABLE = False

#Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

#Configuration for Weights and Thresholds
DEFAULT_CONFIG = {
    "models": {
        "embedding": "all-mpnet-base-v2",
        "spacy": "en_core_web_lg",
    },
    "score_weights": {
        "semantic": 0.40,
        "skill": 0.45,
        "experience": 0.15,
    },
    "grading_thresholds": {
        "priority_review": 80.0,
        "review": 60.0,
        "consider": 45.0,
        "auto_reject": 0.0,
    },
    "experience": {
        "mismatch_penalty_factor": 0.8, # Max score if experience is less than required
    },
    "skill_extraction": {
        "source_weights": {
             "skill_ner": 1.0,
           "regex_acronym": 0.9,
            "regex_tech": 0.8,
            "nlp_noun_chunk": 0.6,
        }
    }
}


class SkillProfile:
    """
    A sophisticated profile to manage, count, and weight skills from a document.
    Replaces the original SkillTree with a more powerful and accurate structure.
    """
    def __init__(self, source_weights: Dict[str, float]):
        self._skills = defaultdict(lambda: {'count': 0, 'sources': set()})
        self.source_weights = source_weights
        self.total_skill_occurrences = 0

    def add(self, skill: str, source: str):
        """Adds a skill, tracking its source for weighted scoring."""
        # Normalize skill for consistency
        skill_clean = re.sub(r'\s+', ' ', skill.lower().strip())
        if len(skill_clean) > 1 and not skill_clean.isdigit():
            self._skills[skill_clean]['count'] += 1
            self._skills[skill_clean]['sources'].add(source)
            self.total_skill_occurrences += 1

    def get_skills(self) -> Set[str]:
        """Returns a set of all unique skills."""
        return set(self._skills.keys())

    def get_weighted_skill_score(self, skill: str) -> float:
        """Calculates a confidence score for a skill based on its sources."""
        skill_info = self._skills.get(skill.lower().strip())
        if not skill_info:
            return 0.0

        # The score is the maximum weight of any of its identified sources.
        max_source_weight = 0.0
        for source in skill_info['sources']:
            max_source_weight = max(max_source_weight, self.source_weights.get(source, 0.0))
        return max_source_weight

    def get_top_skills(self, n: int = 10) -> List[Tuple[str, int]]:
        """Returns the most frequently mentioned skills."""
        # Ensure the items are sorted correctly by count (second element of the tuple)
        return sorted(self._skills.items(), key=lambda item: item[1]['count'], reverse=True)[:n]

    @staticmethod
    def calculate_skill_coverage(candidate_profile: 'SkillProfile', job_profile: 'SkillProfile') -> float:
        """
        Calculates how well the candidate's skills cover the job's required skills.
        This is the most critical skill metric: of the skills the job wants, how many does the candidate have?

        Returns:
            A score from 0.0 to 1.0.
        """
        job_skills = job_profile.get_skills()
        if not job_skills:
            return 1.0  # If job requires no specific skills, coverage is perfect.

        candidate_skills = candidate_profile.get_skills()

        total_weighted_score = 0.0
        max_possible_score = 0.0

        for skill in job_skills:
            # The importance of a skill is determined by its confidence score in the job description.
            job_skill_weight = job_profile.get_weighted_skill_score(skill)
            max_possible_score += job_skill_weight

            if skill in candidate_skills:
                # If the candidate has the skill, they get points equal to the skill's importance.
                total_weighted_score += job_skill_weight

        return total_weighted_score / max_possible_score if max_possible_score > 0 else 0.0


class Cadence:
    """
    An advanced recruitment matching and grading system.
    This class handles text processing, skill extraction, and principled scoring
    to rank candidates for jobs or jobs for candidates.
    """
    def __init__(self, config: Dict[str, Any] = None):
        """
        Initializes the assistant with models and configuration.

        Args:
            config (Dict, optional): A dictionary to override default settings.
        """
        self.config = config or DEFAULT_CONFIG

        logger.info("Initializing Recruitment Assistant...")
        # TODO: For production, consider caching model loading or using a service
        self.model = SentenceTransformer(self.config["models"]["embedding"])
        self.nlp, self.skill_extractor = self._load_nlp_models()
        self.stop_words = self._get_stopwords()

    def _load_nlp_models(self) -> Tuple[Optional[spacy.Language], Optional['SkillExtractor']]:
        """Loads SpaCy and Skill-NER models safely."""
        nlp = None
        skill_extractor = None
        try:
            nlp = spacy.load(self.config["models"]["spacy"])
            logger.info(f"SpaCy model '{self.config['models']['spacy']}' loaded successfully.")
        except OSError:
            logger.warning(f"SpaCy model '{self.config['models']['spacy']}' not found. "
                           f"Run: python -m spacy download {self.config['models']['spacy']}. "
                           "NLP-based skill extraction will be disabled.")

        if nlp and SKILL_NER_AVAILABLE:
            try:
                # TODO: Ensure 'data/skills_db_light.json' exists and is in the correct format for SkillExtractor
                # The error "'dict' object is not callable" suggests an issue with SkillExtractor initialization.
                # Check the SkillExtractor documentation for correct arguments.
                # Assuming the second argument should be the path and the third is configuration/options.
                # If the file is missing or malformed, it could also cause issues.
                # Correcting SkillExtractor initialization
                skill_extractor = SkillExtractor(nlp, "data/skills_db_light.json")
                logger.info("Skill-NER initialized successfully.")
            except Exception as e:
                logger.warning(f"Failed to initialize Skill-NER: {e}. It will be disabled.")

        return nlp, skill_extractor

    def _get_stopwords(self) -> Set[str]:
        """Creates a comprehensive set of stopwords."""
        # Start with SpaCy's default list if available
        stopwords = self.nlp.Defaults.stop_words if self.nlp else set()
        # Add common resume/job description words that are not skills and extracted non-skills
        custom_stopwords = {
            'experience', 'work', 'working', 'job', 'position', 'role', 'responsibilities',
            'duties', 'company', 'team', 'project', 'candidate', 'application', 'etc',
            'e g', 'i e', 'year', 'years', 'month', 'months', 'description', 'requirements',
            # Added based on previous extraction results
            'iit', 'gpa', 'ui', 'api', 'json', 'cd', 'ci', 'saas'
        }
        return stopwords.union(custom_stopwords)

    def _clean_text(self, text: str) -> str:
        """Prepares text for embedding and skill extraction."""
        if not text:
            return ""
        # Remove special characters but preserve ones important for skill names (e.g., C++, .NET, Node.js)
        text = re.sub(r'[^\w\s.+#-]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def extract_skills(self, text: str) -> SkillProfile:
        """
        Extracts skills from text using a multi-method pipeline.

        The pipeline prioritizes higher-confidence methods:
        1. Skill-NER (if available)
        2. Regex for specific technical patterns
        3. SpaCy for general noun phrases
        """
        profile = SkillProfile(self.config["skill_extraction"]["source_weights"])
        clean_text = self._clean_text(text)

        # 1. Skill-NER (Highest Confidence)
        if self.skill_extractor and self.nlp: # Check if nlp is available
            try:
                # SkillNer expects a spacy Doc object, not raw text
                doc = self.nlp(clean_text)
                annotations = self.skill_extractor.annotate(doc) # Pass the Doc object
                for match in annotations['results'].get('full_matches', []):
                    # Skill id might be a tuple or list, convert to string if needed
                    skill_id = match['skill_id']
                    if isinstance(skill_id, (tuple, list)):
                        skill_id = ", ".join(map(str, skill_id))
                    profile.add(skill_id, 'skill_ner')
            except Exception as e:
                logger.warning(f"Skill-NER annotation failed: {e}")

        # 2. Regex-based extraction (Medium Confidence)
        skill_patterns = {
            # e.g., AWS, GCP, API, SQL (avoids short, common acronyms like 'A')
            "regex_acronym": r'\b[A-Z]{3,}\b',
            # e.g., Node.js, .NET, C++, C#
            "regex_tech": r'\b\w+[.#+]{1,2}\w*\b|\b[A-Z][a-z]*\d?[A-Z][a-z]*\b'
        }
        for source, pattern in skill_patterns.items():
            for match in re.findall(pattern, text):
                if match.lower() not in self.stop_words:
                    profile.add(match, source)

        # 3. NLP-based extraction (Lower Confidence)
        if self.nlp:
            doc = self.nlp(clean_text)
            for chunk in doc.noun_chunks:
                chunk_text = chunk.text.lower()
                if len(chunk_text.split()) > 1 and chunk_text not in self.stop_words:
                    profile.add(chunk.text, "nlp_noun_chunk")

        return profile

    def _extract_years_experience(self, text: str) -> float:
        """
        Extracts the maximum years of experience mentioned in the text.
        Handles numbers, words ("five"), and ranges ("3-5").
        """
        text = text.lower()

        number_words = {
            'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5,
            'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10
        }
        for word, num in number_words.items():
            text = text.replace(word, str(num))

        # Pattern to find numbers (digits, possibly with a decimal) followed by "year" or "yr"
        # Handles ranges like "3-5" by capturing the first number
        pattern = r'(\d+(?:\.\d+)?)\s*(?:-|to)?\s*\d*\s*(?:plus|\+)?\s*y(?:ea)?rs?'
        matches = re.findall(pattern, text)

        if not matches:
            return 0.0

        # TODO: Improve experience extraction robustness to avoid picking up years in unrelated contexts
        return max([float(m) for m in matches])

    def _calculate_experience_score(self, candidate_years: float, required_years: float) -> float:
        """Calculates a score from 0.0 to 1.0 based on experience match."""
        if required_years == 0:
            return 1.0  # No experience required, so it's a perfect match.
        if candidate_years >= required_years:
            return 1.0

        # Candidate has less experience than required. Score is proportional, but capped.
        ratio = candidate_years / required_years
        return min(ratio, self.config["experience"]["mismatch_penalty_factor"])

    def _grade_match(self, scores: Dict[str, float]) -> Dict[str, Any]:
        """
        Calculates the final overall score and provides a recommendation.
        """
        weights = self.config["score_weights"]

        # Calculate final weighted score
        overall_score = (
            scores.get('semantic', 0.0) * weights['semantic'] +
            scores.get('skill', 0.0) * weights['skill'] +
            scores.get('experience', 0.0) * weights['experience']
        ) * 100 # Convert to percentage

        # Determine recommendation based on thresholds
        thresholds = self.config["grading_thresholds"]
        if overall_score >= thresholds["priority_review"]:
            action = "PRIORITY_REVIEW"
            feedback = "Excellent match across all key areas. A top candidate."
        elif overall_score >= thresholds["review"]:
            action = "REVIEW"
            feedback = "Strong match. Meets most requirements and is worth reviewing."
        elif overall_score >= thresholds["consider"]:
            action = "CONSIDER"
            feedback = "Partial match. Lacks in some areas but could be considered."
        else:
            action = "AUTO_REJECT"
            feedback = "Poor match. Does not meet key requirements for the role."

        # Add more specific feedback
        # Note: The conditions here are just examples and can be refined
        if scores['experience'] < 0.8 and scores['experience'] < scores['skill']:
             feedback += " The primary gap is years of experience."
        elif scores['skill'] < 0.7: # Threshold for skill gap emphasis
             feedback += " The primary gap is in the required skills."
        elif scores['semantic'] < 0.6: # Threshold for semantic gap emphasis
             feedback += " The resume's overall content doesn't strongly align."


        return {
            "overall_score": round(overall_score, 2),
            "action": action,
            "feedback": feedback
        }

    def _create_match(self, source_doc: Dict, target_doc: Dict) -> Dict[str, Any]:
        """
        The core matching engine. Computes all scores between a source and a target.
        This internal method eliminates code duplication.

        Args:
            source_doc: The document to match from (e.g., a resume).
            target_doc: The document to match against (e.g., a job).
        """
        # 1. Score Calculation
        # Semantic Score (Cosine Similarity)
        semantic_score = util.cos_sim(source_doc['embedding'], target_doc['embedding']).item()

        # Skill Score (Weighted Coverage)
        skill_score = SkillProfile.calculate_skill_coverage(
            candidate_profile=source_doc['skill_profile'],
            job_profile=target_doc['skill_profile']
        )

        # Experience Score
        experience_score = self._calculate_experience_score(
            candidate_years=source_doc['experience_years'],
            required_years=target_doc['experience_years']
        )

        component_scores = {
            "semantic": semantic_score,
            "skill": skill_score,
            "experience": experience_score
        }

        # 2. Grading
        grading_result = self._grade_match(component_scores)

        # 3. Result Assembly
        common_skills = source_doc['skill_profile'].get_skills().intersection(target_doc['skill_profile'].get_skills())

        return {
            "recommendation": {
                "action": grading_result["action"],
                "overall_score": grading_result["overall_score"],
                "feedback": grading_result["feedback"],
            },
            "scores": {
                "semantic_percent": round(semantic_score * 100, 2),
                "skill_coverage_percent": round(skill_score * 100, 2),
                "experience_match_percent": round(experience_score * 100, 2),
            },
            "details": {
                "experience": {
                    "candidate": source_doc['experience_years'],
                    "required": target_doc['experience_years'],
                },
                "skills": {
                    "common_skills": list(common_skills),
                    # TODO: Add confidence scores to job and candidate skills in output for better detail
                    "job_requirements": target_doc['skill_profile'].get_top_skills(),
                    "candidate_skills": source_doc['skill_profile'].get_top_skills(),
                }
            }
        }

    def rank_jobs_for_resume(self, resume_content: str, job_descriptions: List[str], top_k: int = 5) -> List[Dict[str, Any]]:
        """
        Finds and ranks the best job matches for a given resume.
        """
        logger.info(f"Processing resume and {len(job_descriptions)} job descriptions...")
        resume_text = self._clean_text(resume_content)
        resume_doc = {
            "text": resume_text,
            # TODO: Implement caching of embeddings for performance with large datasets
            "embedding": self.model.encode(resume_text),
            "skill_profile": self.extract_skills(resume_text),
            "experience_years": self._extract_years_experience(resume_text),
        }

        job_docs = []
        for job_text in job_descriptions:
            clean_job_text = self._clean_text(job_text)
            job_docs.append({
                "text": clean_job_text,
                # TODO: Implement caching of embeddings for performance with large datasets
                "embedding": self.model.encode(clean_job_text),
                "skill_profile": self.extract_skills(clean_job_text),
                "experience_years": self._extract_years_experience(clean_job_text),
            })

        results = []
        for i, job_doc in enumerate(job_docs):
            match_result = self._create_match(source_doc=resume_doc, target_doc=job_doc)
            match_result['job_index'] = i
            results.append(match_result)

        # TODO: For very large numbers of jobs, consider using approximate nearest neighbor search (e.g., FAISS)
        # for the semantic similarity part instead of calculating pairwise similarity with all jobs.
        # Sort by the final, principled overall score
        results.sort(key=lambda x: x['recommendation']['overall_score'], reverse=True)
        return results[:top_k]

    def rank_resumes_for_job(self, job_description: str, resume_contents: List[str], top_k: int = 10) -> List[Dict[str, Any]]:
        """
        Finds and ranks the best resume matches for a given job description.
        """
        logger.info(f"Processing job description and {len(resume_contents)} resumes...")
        job_text = self._clean_text(job_description)
        job_doc = {
            "text": job_text,
            # TODO: Implement caching of embeddings for performance with large datasets
            "embedding": self.model.encode(job_text),
            "skill_profile": self.extract_skills(job_text),
            "experience_years": self._extract_years_experience(job_text),
        }

        resume_docs = []
        for resume_text in resume_contents:
            clean_resume_text = self._clean_text(resume_text)
            resume_docs.append({
                "text": clean_resume_text,
                 # TODO: Implement caching of embeddings for performance with large datasets
                "embedding": self.model.encode(clean_resume_text),
                "skill_profile": self.extract_skills(clean_resume_text),
                "experience_years": self._extract_years_experience(clean_resume_text), # Changed from clean_text to clean_resume_text
            })

        results = []
        for i, resume_doc in enumerate(resume_docs):
            # Note the order: source is resume, target is job
            match_result = self._create_match(source_doc=resume_doc, target_doc=job_doc)
            match_result['resume_index'] = i
            results.append(match_result)

        # TODO: For very large numbers of resumes, consider using approximate nearest neighbor search (e.g., FAISS)
        # for the semantic similarity part instead of calculating pairwise similarity with all resumes.
        results.sort(key=lambda x: x['recommendation']['overall_score'], reverse=True)
        return results[:top_k]

    def analyze_skill_gap(self, resume_content: str, job_description: str) -> Dict[str, Any]:
        """Provides a detailed analysis of skill gaps between a resume and a job."""
        resume_profile = self.extract_skills(resume_content)
        job_profile = self.extract_skills(job_description)

        resume_skills = resume_profile.get_skills()
        job_skills = job_profile.get_skills()

        missing_skills = job_skills - resume_skills

        # Rank missing skills by their importance in the job description
        ranked_missing = sorted(
            list(missing_skills),
            key=lambda skill: job_profile.get_weighted_skill_score(skill),
            reverse=True
        )

        return {
            "skill_coverage_percent": round(SkillProfile.calculate_skill_coverage(resume_profile, job_profile) * 100, 2),
            "missing_skills_from_job": ranked_missing,
            "matching_skills": list(resume_skills.intersection(job_skills)),
            "candidate_unique_skills": list(resume_skills - job_skills),
        }

In [ ]:
if __name__ == '__main__':
    cadence = Cadence()

    # --- Candidate 1: The Experienced Senior Data Engineer ---
    resume_senior_data_engineer = """
    Priya Sharma
    Senior Data Engineer
    San Francisco, CA | (555) 123-4567 | priya.sharma@email.com | linkedin.com/in/priyasharma-data

    --- Summary ---
    Accomplished and results-oriented Senior Data Engineer with 9+ years of experience in designing, building, and optimizing large-scale data pipelines and data warehousing solutions. Expert in leveraging big data technologies like Apache Spark and Kafka to process terabytes of data efficiently. Proven ability to lead data projects, mentor junior engineers, and collaborate with cross-functional teams to deliver data-driven insights.

    --- Experience ---
    DataCorp Analytics, San Francisco, CA | Senior Data Engineer (Jan 2019 - Present)
    - Architected and implemented a new event-driven data ingestion pipeline using Kafka, Spark Streaming, and Python, processing over 2 billion events daily with 99.9% uptime.
    - Designed and managed the enterprise data warehouse on Amazon Redshift, reducing query latency for BI tools by 40%.
    - Developed and maintained complex ETL/ELT jobs using Apache Airflow, ensuring data quality and reliability for critical business reporting.
    - Led a project to migrate legacy data systems to a modern cloud data lake on AWS S3 and AWS Glue, cutting infrastructure costs by 25%.
    - Mentored a team of 3 junior data engineers on best practices for data modeling and performance tuning.

    Analytics Inc., San Jose, CA | Data Engineer (June 2015 - Dec 2018)
    - Developed Python scripts to automate data extraction from various APIs and relational databases (PostgreSQL, MySQL).
    - Built data models and dashboards in Tableau to support marketing and sales teams.
    - Optimized complex SQL queries, improving report generation times by over 50%.

    --- Education ---
    M.S. in Computer Science, Stanford University (2015)
    B.Tech in Information Technology, IIT Delhi (2013)

    --- Skills ---
    - Programming Languages: Python (Expert), SQL (Expert), Scala (Proficient), Java (Intermediate)
    - Big Data Technologies: Apache Spark, Apache Kafka, Apache Airflow, Hadoop, HDFS
    - Cloud Platforms (AWS): Redshift, S3, Glue, EMR, Lambda, EC2, IAM
    - Databases: PostgreSQL, MySQL, Amazon Redshift, NoSQL (MongoDB, DynamoDB)
    - Data Warehousing: Dimensional Modeling, ETL/ELT design, Data Lake architecture
    - Tools: Docker, Git, Jenkins (CI/CD), Terraform (IaC)
    """

    # Candidate 2: The Eager Recent Graduate
    resume_recent_graduate = """
    Alex Chen
    (123) 456-7890 | alex.chen.dev@email.com | github.com/alexchen

    --- Objective ---
    A highly motivated and passionate recent Computer Science graduate with a solid foundation in software development and a specialization in full-stack web technologies. Seeking an entry-level Software Engineer position to apply my skills in JavaScript, React, and Node.js to build innovative products and grow within a collaborative team.

    --- Education ---
    University of California, Berkeley | B.S. in Computer Science (GPA: 3.8/4.0)
    Graduated: May 2024
    - Relevant Coursework: Data Structures & Algorithms, Web Development, Database Systems, Operating Systems

    --- Projects ---
    EventFinder | MERN Stack Web Application (https://github.com/alexchen/eventfinder)
    - Developed a full-stack web application to discover and RSVP to local events.
    - Built a RESTful API backend using Node.js, Express.js, and MongoDB for data management.
    - Designed a responsive and interactive user interface with React, Redux for state management, and Material-UI.
    - Implemented user authentication and authorization using JSON Web Tokens (JWT).

    AlgoVisualizer | Python & Pygame Application
    - Created a desktop tool to visualize common sorting and pathfinding algorithms (e.g., A*, Dijkstra's).
    - Implemented algorithms from scratch in Python to deepen understanding of their mechanics.

    --- Skills ---
    - Languages: JavaScript (ES6+), Python, HTML5, CSS3, SQL
    - Frontend: React, Redux, HTML, CSS, Bootstrap, Material-UI
    - Backend: Node.js, Express.js
    - Databases: MongoDB, PostgreSQL (Basic)
    - Tools: Git, npm, Webpack, Babel, Docker (Basic)
    """

    # Job 1: Highly Specific Senior Data Engineer Role
    job_senior_data_engineer = """
    Senior Data Engineer - QuantumLeap Analytics

    About Us: At QuantumLeap Analytics, we are at the forefront of AI-driven business intelligence. We're looking for a seasoned Senior Data Engineer to join our core platform team and help us build the data infrastructure that powers our insights.

    The Role: You will be a technical leader responsible for designing, building, and scaling our mission-critical data pipelines. This is a hands-on role that requires deep technical expertise and a passion for data.

    Responsibilities:
    - Design, construct, and maintain robust and scalable ETL/ELT pipelines.
    - Architect our next-generation data lake and data warehousing solutions on AWS.
    - Optimize data processing jobs written in Python and Spark for performance and cost.
    - Work with data scientists to productionize machine learning models.
    - Mentor junior engineers and champion data engineering best practices.

    Must-Have Qualifications:
    - 7+ years of hands-on experience as a Data Engineer.
    - Expert-level proficiency in Python and advanced SQL.
    - 4+ years of production experience with Apache Spark.
    - Deep, hands-on experience with the AWS data ecosystem (Redshift, S3, Glue, EMR, Kinesis/Kafka).
    - Strong experience with workflow orchestration tools like Apache Airflow.
    - A degree in Computer Science, Engineering, or a related field.

    Nice-to-Have Qualifications:
    - Experience with Infrastructure as Code (Terraform, CloudFormation).
    - Knowledge of data modeling tools like dbt.
    - Experience with streaming data technologies beyond ingestion (e.g., Flink).
    """

    # Job 2: Mid-Level Full-Stack Role
    job_full_stack_engineer = """
    Full-Stack Software Engineer - Innovate Solutions

    Innovate Solutions is building a revolutionary SaaS platform for project management. We're hiring a talented Full-Stack Engineer to help us build new end-to-end features.

    What You'll Do:
    - Develop beautiful, responsive user-facing features using React and TypeScript.
    - Build, test, and deploy robust backend APIs and services with Node.js.
    - Collaborate with product managers and designers to turn ideas into reality.
    - Write clean, maintainable, and well-tested code.
    - Participate in our agile development process.

    Requirements:
    - 3-5 years of professional software development experience.
    - Strong proficiency in JavaScript/TypeScript and the Node.js ecosystem.
    - Extensive hands-on experience with React and modern state management libraries (e.g., Redux, Zustand).
    - Experience designing and consuming RESTful APIs.
    - Solid understanding of both SQL (e.g., PostgreSQL) and NoSQL (e.g., MongoDB) databases.
    - Familiarity with Git and CI/CD principles.
    """

    # RUNNING DETAILED TEST SCENARIOS

    print("\n" + "="*20 + " SCENARIO 1: Ranking Candidates for the SENIOR DATA ENGINEER Job " + "="*20)
    candidates_for_data_job = cadence.rank_resumes_for_job(
        job_description=job_senior_data_engineer,
        resume_contents=[resume_senior_data_engineer, resume_recent_graduate]
    )
    for i, match in enumerate(candidates_for_data_job):
        candidate_name = "Priya Sharma (Senior)" if match['resume_index'] == 0 else "Alex Chen (Graduate)"
        print(f"\n--- Result {i+1}: {candidate_name} ---")
        print(f"  Overall Score: {match['recommendation']['overall_score']}% | Action: {match['recommendation']['action']}")
        print(f"  Feedback: {match['recommendation']['feedback']}")
        print(f"  Scores Breakdown -> Skill: {match['scores']['skill_coverage_percent']}% | Experience: {match['scores']['experience_match_percent']}% | Semantic: {match['scores']['semantic_percent']}%")
        print(f"  Experience Match: Candidate has ~{match['details']['experience']['candidate']} years, Job requires ~{match['details']['experience']['required']} years.")

        # Skill gap analysis
        gap_analysis = cadence.analyze_skill_gap(
            resume_content=resume_senior_data_engineer if match['resume_index'] == 0 else resume_recent_graduate,
            job_description=job_senior_data_engineer
        )
        print(f"  Top Missing Skills: {gap_analysis['missing_skills_from_job'][:3]}")



    print("\n\n" + "="*20 + " SCENARIO 2: Ranking Candidates for the FULL-STACK ENGINEER Job " + "="*20)
    candidates_for_fs_job = cadence.rank_resumes_for_job(
        job_description=job_full_stack_engineer,
        resume_contents=[resume_senior_data_engineer, resume_recent_graduate]
    )
    for i, match in enumerate(candidates_for_fs_job):
        candidate_name = "Priya Sharma (Senior)" if match['resume_index'] == 0 else "Alex Chen (Graduate)"
        print(f"\n--- Result {i+1}: {candidate_name} ---")
        print(f"  Overall Score: {match['recommendation']['overall_score']}% | Action: {match['recommendation']['action']}")
        print(f"  Feedback: {match['recommendation']['feedback']}")
        print(f"  Scores Breakdown -> Skill: {match['scores']['skill_coverage_percent']}% | Experience: {match['scores']['experience_match_percent']}% | Semantic: {match['scores']['semantic_percent']}%")
        print(f"  Experience Match: Candidate has ~{match['details']['experience']['candidate']} years, Job requires ~{match['details']['experience']['required']} years.")

        gap_analysis = cadence.analyze_skill_gap(
            resume_content=resume_senior_data_engineer if match['resume_index'] == 0 else resume_recent_graduate,
            job_description=job_full_stack_engineer
        )
        print(f"  Top Missing Skills: {gap_analysis['missing_skills_from_job'][:3]}")


==================== SCENARIO 1: Ranking Candidates for the SENIOR DATA ENGINEER Job ====================

--- Result 1: Priya Sharma (Senior) ---
  Overall Score: 55.45% | Action: CONSIDER
  Feedback: Partial match. Lacks in some areas but could be considered. The primary gap is in the required skills.
  Scores Breakdown -> Skill: 20.6% | Experience: 100.0% | Semantic: 77.96%
  Experience Match: Candidate has ~9.0 years, Job requires ~7.0 years.
  Top Missing Skills: ['ai', 'quantumleap', 'cloudformation']

--- Result 2: Alex Chen (Graduate) ---
  Overall Score: 29.54% | Action: AUTO_REJECT
  Feedback: Poor match. Does not meet key requirements for the role. The primary gap is years of experience.
  Scores Breakdown -> Skill: 5.07% | Experience: 0.0% | Semantic: 68.13%
  Experience Match: Candidate has ~0.0 years, Job requires ~7.0 years.
  Top Missing Skills: ['elt', 'emr', 'aws']


==================== SCENARIO 2: Ranking Candidates for the FULL-STACK ENGINEER Job =================